<a href="https://colab.research.google.com/github/Jheeshith/24ai049-Jheeshith3/blob/main/CAT_2_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install kagglehub pandas

import kagglehub

path = kagglehub.dataset_download(
    "yash9439/eng-hinglish-machine-translation"
)

print("Dataset downloaded to:", path)

100%|██████████| 476k/476k [00:00<00:00, 62.7MB/s]

Extracting files...
Dataset downloaded to: /root/.cache/kagglehub/datasets/yash9439/eng-hinglish-machine-translation/versions/1


In [4]:
import os
import zipfile
import pandas as pd

# Extract ZIP files if present
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(".zip"):
            zip_path = os.path.join(root, file)
            extract_path = os.path.join(path, "extracted")

            os.makedirs(extract_path, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extract_path)

# Find files by name, including nested folders
def find_file(filename):
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.lower() == filename.lower():
                return os.path.join(root, file)
    raise FileNotFoundError(
        f"{filename} not found. Check the output of Cell 1."
    )

def load_pairs(filename):
    filepath = find_file(filename)

    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["english", "hinglish"],
        dtype=str,
        on_bad_lines="warn"
    )

    return df.dropna().reset_index(drop=True)

train_df = load_pairs("train.txt")
val_df = load_pairs("dev.txt")
test_df = load_pairs("test.txt")

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

display(train_df.head())

Training: 7040
Validation: 1020
Testing: 942


,english,hinglish
0,Batman vs Superman,batman vs superman
1,"The director is Zack Snyder , 27 % Rotten Toma...","Zack Snyder director हैं , 27 % Rotten Tomatoe..."
2,Not very popular it seems,लगता है बहुत popular नहीं है
3,But the audiences liked it . It has a B cinema...,"but audience ने like किया , इसका cinema score ..."
4,Yes,yes


In [10]:

import os

print("Dataset folder:", path)
print("\nFiles found inside the dataset:")

for root, dirs, files in os.walk(path):
    print("Folder:", root)
    for file in files:
        print("  ", file)


Dataset folder: /root/.cache/kagglehub/datasets/yash9439/eng-hinglish-machine-translation/versions/1

Files found inside the dataset:
Folder: /root/.cache/kagglehub/datasets/yash9439/eng-hinglish-machine-translation/versions/1
Folder: /root/.cache/kagglehub/datasets/yash9439/eng-hinglish-machine-translation/versions/1/Eng - Hinglish
   test.txt
   train.txt
   dev.txt


In [12]:
import pandas as pd
import os

def load_pairs(filename):
    # The files are inside the 'Eng - Hinglish' subdirectory
    filepath = os.path.join(path, "Eng - Hinglish", filename)

    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["english", "hinglish"],
        dtype=str,
        on_bad_lines="warn"
    )

    return df.dropna().reset_index(drop=True)

train_df = load_pairs("train.txt")
val_df   = load_pairs("dev.txt")
test_df  = load_pairs("test.txt")

print("Training pairs:", len(train_df))
print("Validation pairs:", len(val_df))
print("Testing pairs:", len(test_df))

display(train_df.head(10))

Training pairs: 7040
Validation pairs: 1020
Testing pairs: 942


,english,hinglish
0,Batman vs Superman,batman vs superman
1,"The director is Zack Snyder , 27 % Rotten Toma...","Zack Snyder director हैं , 27 % Rotten Tomatoe..."
2,Not very popular it seems,लगता है बहुत popular नहीं है
3,But the audiences liked it . It has a B cinema...,"but audience ने like किया , इसका cinema score ..."
4,Yes,yes
5,There is a huge divergence between proffession...,huge divergence है profession critic की opinio...
6,I've never seen it,मैंने तो कभी नहीं देखा है .
7,I know the difference .,मुझे difference पाता है
8,I can't believe they used Ben Affleck as Batman,मुझे तो believe ही नहीं होता कि उन्होंने Ben A...
9,"So it probably won't get any awards , but shou...","So Probably इसको कोई award नहीं मिला , but box..."


In [13]:
for i in range(min(5, len(train_df))):
    print("English :", train_df.iloc[i]["english"])
    print("Hinglish:", train_df.iloc[i]["hinglish"])
    print("-" * 50)

English : Batman vs Superman
Hinglish: batman vs superman 
--------------------------------------------------
English : The director is Zack Snyder , 27 % Rotten Tomatoes , 4.9 / 10 .
Hinglish: Zack Snyder director हैं , 27 % Rotten Tomatoes , 4.9 / 10 . 
--------------------------------------------------
English : Not very popular it seems
Hinglish: लगता है बहुत popular नहीं है 
--------------------------------------------------
English : But the audiences liked it . It has a B cinema score
Hinglish: but audience ने like किया , इसका cinema score भी है 
--------------------------------------------------
English : Yes
Hinglish: yes 
--------------------------------------------------


In [14]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("All CSV files created successfully.")

All CSV files created successfully.


In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer

SOS = "<sos>"
EOS = "<eos>"

train_df["target"] = SOS + " " + train_df["hinglish"] + " " + EOS
val_df["target"] = SOS + " " + val_df["hinglish"] + " " + EOS
test_df["target"] = SOS + " " + test_df["hinglish"] + " " + EOS

src_tokenizer = Tokenizer(oov_token="<unk>", filters="")
tgt_tokenizer = Tokenizer(oov_token="<unk>", filters="")

src_tokenizer.fit_on_texts(train_df["english"])
tgt_tokenizer.fit_on_texts(train_df["target"])

src_vocab_size = len(src_tokenizer.word_index) + 1
tgt_vocab_size = len(tgt_tokenizer.word_index) + 1

print("English vocabulary:", src_vocab_size)
print("Hinglish vocabulary:", tgt_vocab_size)

English vocabulary: 5655
Hinglish vocabulary: 7680


In [18]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = src_tokenizer.texts_to_sequences(train_df["english"])
Y_train = tgt_tokenizer.texts_to_sequences(train_df["target"])

X_val = src_tokenizer.texts_to_sequences(val_df["english"])
Y_val = tgt_tokenizer.texts_to_sequences(val_df["target"])

X_test = src_tokenizer.texts_to_sequences(test_df["english"])
Y_test = tgt_tokenizer.texts_to_sequences(test_df["target"])

max_src_len = max(map(len, X_train))
max_tgt_len = max(map(len, Y_train))

X_train = pad_sequences(X_train, maxlen=max_src_len, padding="post")
X_val = pad_sequences(X_val, maxlen=max_src_len, padding="post")
X_test = pad_sequences(X_test, maxlen=max_src_len, padding="post")

Y_train = pad_sequences(Y_train, maxlen=max_tgt_len, padding="post")
Y_val = pad_sequences(Y_val, maxlen=max_tgt_len, padding="post")
Y_test = pad_sequences(Y_test, maxlen=max_tgt_len, padding="post")

decoder_input_train = Y_train[:, :-1]
decoder_target_train = Y_train[:, 1:]

decoder_input_val = Y_val[:, :-1]
decoder_target_val = Y_val[:, 1:]

decoder_input_test = Y_test[:, :-1]
decoder_target_test = Y_test[:, 1:]

print("Encoder input:", X_train.shape)
print("Decoder input:", decoder_input_train.shape)
print("Decoder target:", decoder_target_train.shape)

Encoder input: (7040, 283)
Decoder input: (7040, 372)
Decoder target: (7040, 372)


In [20]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

embedding_dim = 128
lstm_units = 256

encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(
    src_vocab_size, embedding_dim, mask_zero=True
)(encoder_inputs)

_, state_h, state_c = LSTM(
    lstm_units, return_state=True
)(encoder_embedding)

encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_embedding_layer = Embedding(
    tgt_vocab_size, embedding_dim, mask_zero=True
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    lstm_units, return_sequences=True, return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding, initial_state=encoder_states
)

decoder_dense = Dense(tgt_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs], decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    723,840 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │    983,040 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  1,973,760 │ lstm_1[0][0]      │
│                     │ 7680)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,469,120 (17.05 MB)

 Trainable params: 4,469,120 (17.05 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    validation_data=(
        [X_val, decoder_input_val],
        decoder_target_val
    ),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/20
101/220 ━━━━━━━━━━━━━━━━━━━━ 16:41 8s/step - loss: 7.3576 - sparse_categorical_accuracy: 0.0893

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_h = Input(shape=(lstm_units,))
decoder_state_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_h, decoder_state_c]

decoder_single_input = Input(shape=(1,))
decoder_single_embedding = decoder_embedding_layer(
    decoder_single_input
)

decoder_single_output, state_h, state_c = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs
)

decoder_single_output = decoder_dense(decoder_single_output)

decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_single_output, state_h, state_c]
)